In [1]:
!pip install --no-deps git+https://github.com/ai4bharat/IndicF5.git

  Cloning https://github.com/ai4bharat/IndicF5.git to /tmp/pip-req-build-ygpo1le2
  Running command git clone --filter=blob:none --quiet https://github.com/ai4bharat/IndicF5.git /tmp/pip-req-build-ygpo1le2
  Resolved https://github.com/ai4bharat/IndicF5.git to commit 13f7c4d627cc10111aea8fe9c0039462cacacdc7
  Preparing metadata (setup.py) ... done
  Created wheel for f5_tts: filename=f5_tts-0.1.0-py3-none-any.whl size=63528 sha256=e1f44fe2cde4b4a346e4a03e33848cc131d9d287446ffd1a67393ff877e98418
  Stored in directory: /tmp/pip-ephem-wheel-cache-t_nc_6mt/wheels/b8/d6/ca/0aa683aabe33e0e410f96fd083518d97ea4d1317a425879b37
Successfully built f5_tts


In [2]:
packages = [
    "transformers==4.49.0", "vocos", "x_transformers", "torchdiffeq",
    "ema_pytorch", "cached_path", "accelerate", "hydra-core", "jieba",
    "librosa", "pydub", "pypinyin", "transformers_stream_generator", "soundfile",
]

for pkg in packages:
    print("Installing", pkg)
    !pip install -q {pkg}

print("Install finished")

Installing transformers==4.49.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 61.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
f5-tts 0.1.0 requires cached_path, which is not installed.
f5-tts 0.1.0 requires ema_pytorch>=0.5.2, which is not installed.
f5-tts 0.1.0 requires hydra-core>=1.3.0, which is not installed.
f5-tts 0.1.0 requires pypinyin, which is not installed.
f5-tts 0.1.0 requires tomli, which is not installed.
f5-tts 0.1.0 requires torchdiffeq, which is not installed.
f5-tts 0.1.0 requires transformers_stream_generator, which is not installed.
f5-tts 0.1.0 requires vocos, which is not

In [3]:
import transformers, numpy, vocos
print("transformers:", transformers.__version__)
print("numpy:", numpy.__version__)
print("vocos OK")

transformers: 4.49.0
numpy: 2.1.3
vocos OK


In [8]:
from google.colab import files
import time
import numpy as np
import soundfile as sf
from IPython.display import Audio, display

uploaded = files.upload()
hindi_sample = list(uploaded.keys())[0]
hindi_sample_text = "मैं वॉइस ट्रांसलेटर को अपनी आवाज़ का इस्तेमाल करने की अनुमति देता हूँ। यह मेरी असली आवाज़ है।"

text = "मुझे नहीं पता कि कहाँ जाना है। कृपया मेरी मदद करें।"
start = time.time()
audio = indic_tts(text, ref_audio_path=hindi_sample, ref_text=hindi_sample_text)
if audio.dtype == np.int16:
    audio = audio.astype(np.float32) / 32768.0
sf.write("hindi_test.wav", np.array(audio, dtype=np.float32), samplerate=24000)
print(f"Done in {time.time() - start:.1f}s")
display(Audio("hindi_test.wav"))

Saving WhatsApp Ptt 2026-09-30 at 11.46.31 PM.ogg to WhatsApp Ptt 2026-09-30 at 11.46.31 PM.ogg
Done in 15.0s


In [10]:
texts = {
    "Hindi": "मुझे नहीं पता कि कहाँ जाना है। कृपया मेरी मदद करें।",
    "Tamil": "எங்கு செல்வது என்று தெரியவில்லை. தயவுசெய்து எனக்கு உதவுங்கள்.",
    "Telugu": "ఎక్కడికి వెళ్ళాలో నాకు తెలియదు. దయచేసి నాకు సహాయం చేయండి.",
}

for name, text in texts.items():
    start = time.time()
    audio = indic_tts(text, ref_audio_path=hindi_sample, ref_text=hindi_sample_text)
    if audio.dtype == np.int16:
        audio = audio.astype(np.float32) / 32768.0
    path = f"{name}_v2.wav"
    sf.write(path, np.array(audio, dtype=np.float32), samplerate=24000)
    print(f"{name}: {time.time() - start:.1f}s")
    display(Audio(path))

Hindi: 15.3s


Tamil: 16.8s


Telugu: 16.7s


In [11]:
# Login and load IndicF5 onto the GPU
from google.colab import userdata
from huggingface_hub import login
from transformers import AutoModel

login(token=userdata.get("HF_TOKEN"))
print("Logged in")

indic_tts = AutoModel.from_pretrained(
    "ai4bharat/IndicF5", trust_remote_code=True, low_cpu_mem_usage=False
).to("cuda")
print("IndicF5 loaded on GPU")

Logged in
Download Vocos from huggingface charactr/vocos-mel-24khz

vocab :  /root/.cache/huggingface/hub/models--ai4bharat--IndicF5/snapshots/ba85abedf18dc479a447eaa0eccbd76ab78a47d5/checkpoints/vocab.txt
token :  custom
IndicF5 loaded on GPU


In [6]:
import torch
print("Model was on:", next(indic_tts.parameters()).device)
indic_tts = indic_tts.to("cuda")
print("Model now on:", next(indic_tts.parameters()).device)

Model was on: cpu
Model now on: cuda:0
